# 9.9 最大 softmax 機率很高就可靠嗎？


文件颜色是紅，三個候選按藍、紅、绿编號0、1、2，分數2、1、0卻偏藍。把全部分數乘10，最高仍藍，相對機率會更尖，但沒有加入紅色事實。

logits是候選分數，softmax把它们转為總和1的比例；最大比例只比较在場候選，不查標准答案。以下correct_index=1是真值，scale只改尺度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

scores = torch.tensor([[2.0, 1.0, 0.0]])
correct_index = 1
for scale in [1.0, 10.0]:
    probability = (scores * scale).softmax(-1)
    confidence, prediction = probability.max(-1)
    print(
        "倍率",
        scale,
        "信心",
        round(confidence.item(), 4),
        "選擇",
        prediction.item(),
        "正確",
        prediction.item() == correct_index,
    )

`torch.tensor`把數列或表格交給PyTorch計算；`[[2.0, 1.0, 0.0]]`的外層放一筆資料，內層放三個候選分數，所以是一筆、三個候選的表格。另用`correct_index=1`設第二項是標準答案。索引從0起，因此最高分的第一項是錯誤候選。程式只改變整體倍率，`softmax(-1)`在最後的候選軸做轉換，`max(-1)`同時取最大比例與其索引。兩者各只剩一個值，`.item()`把它取成普通Python數值，方便交給`round`四捨五入，或與標準索引比較。

倍率1時輸出信心約0.6652、選擇0、正確False；倍率10時輸出信心約1.0、選擇仍0、正確仍False。由於四舍五入，看起來恰好1.0，內部其實仍略小於1。兩次沒有訓練、沒有加入任何事實，變化只來自分數尺度。這正是高信心錯誤的可追蹤例子。

除正溫度再softmax也调整相對尖度，正值不改排名；可用于校准報告概率，不能把錯誤最高項變新知識。若所有候選都不合适，softmax仍分完1，仍有最高項。整段回答是否可靠，還需要許多獨立題的正確率，不是一次最大比例。

練習只把`correct_index`改成0，先預測信心數字完全不變，正確判定兩次都變True，再重跑核對。這個改變來自標準答案，不是模型能力變化。要判斷信心可靠性，必須有許多獨立題與真值，比較比如「信心0.9的題有多少真的正確」，下一節會把這件事算出來。

<details>
<summary>補充：實作約定與原始紀錄</summary>

這件事也出現在真正更新過的模型上。[12.8的單音編碼器](https://birdhackor.github.io/tiny-perceptron-vlm/12.8.html)學的是「大於300Hz為high」，並不是安全政策。它在未訓練的320Hz、振幅0.5、長0.12秒錄音上選了錯誤的low，原始softmax信心約0.7490。只用另一批驗證錄音選出的溫度0.5，把同一組分數除以0.5後，信心升到約0.8990，答案仍是low、仍然錯。正溫度維持分數排名，實驗也逐題核對了最高候選不變；調整的是報出的機率，沒有新增音高知識。

這些數字來自[正式實跑報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/encoders.json)中音訊分類器的兩個固定候選，不能讀成「整段生成回答有89.9%機率正確」，更不能當成模型安全判斷的可靠度。熟練完成某種小任務，與知道何時可靠、何時該保留判斷，是需要不同證據的兩件事。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
